In [1]:
# assign directory
import git
from pathlib import Path
import os
ROOT_DIR = Path(git.Repo('.', search_parent_directories=True).working_tree_dir)
os.chdir(os.path.join(ROOT_DIR, "utilities"))
from transform_audio import *
os.chdir(os.path.join(ROOT_DIR, "dataset-preparation"))

respiratory_dir  = os.path.join(ROOT_DIR, 'raw-data', 'respiratory')
data_dir = os.path.join(respiratory_dir, 'audio_and_txt_files')

### Split audio and text into separate directories, filter sampling rate to 44100

In [10]:
audio_dir = os.path.join(respiratory_dir, 'audio_files')
text_dir = os.path.join(respiratory_dir, 'text_files')

if not (os.path.exists(audio_dir) and os.path.exists(text_dir)):
    os.mkdir(audio_dir)
    os.mkdir(text_dir)

    file_names = [name[:-4] for name in os.listdir(data_dir) if name.endswith('.wav')]
    for name in file_names:
        data_path = os.path.join(data_dir, name)
        if wavfile.read(data_path + '.wav')[0] != 44100:
            continue
        shutil.copy(data_path + '.wav', os.path.join(audio_dir, name) + '.wav')
        shutil.copy(data_path + '.txt', os.path.join(text_dir, name) + '.txt')

file_names = [name[:-4] for name in os.listdir(audio_dir)]
audio_file_list = [os.path.join(audio_dir, name) + '.wav' for name in file_names]
text_file_list = [os.path.join(text_dir, name) + '.txt' for name in file_names]
print(f'{len(file_names)} files remaining')

824 files remaining


### Truncate by respiratory cycle

In [18]:
CYCLES_TO_KEEP = 2 # can be at most 2 for compatibility with all files

def trim_audio_file(input_path, start_time, end_time, output_path):
    rate, signal = wavfile.read(input_path)
    start, end = int(start_time * rate), int(end_time * rate)
    wavfile.write(output_path, rate, signal[start:end])

trimmed_audio_dir = os.path.join(respiratory_dir, 'trimmed_audio_files')
if not os.path.exists(trimmed_audio_dir):
    os.mkdir(trimmed_audio_dir)

    for name, audio, text in zip(file_names, audio_file_list, text_file_list):
        table = pd.read_table(text, header=None)
        start_time = table.iloc[0, 0]
        end_time = table.iloc[CYCLES_TO_KEEP - 1, 1]
        output_path = os.path.join(trimmed_audio_dir, name) + '.wav'
        trim_audio_file(audio, start_time, end_time, output_path)

audio_file_list = [os.path.join(trimmed_audio_dir, name) + '.wav' for name in file_names]